In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import networkx as nx
from scipy.stats import beta, betabinom
import matplotlib.pyplot as plt
from collections import Counter
from study3.config import *
from study3.utils import *

In [2]:
study2 = pd.read_csv(DATA_DIR / "qualtrics_experiments" / "desc_norms_init_norm_pilot_long.csv")

In [3]:
study2.columns

Index(['pid', 'progress', 'duration', 'finished', 'feedback', 'llm_usage',
       'usage_convention', 'usage_moral', 'usage_personal', 'llm_usage_binary',
       'usage_convention_binary', 'usage_moral_binary',
       'usage_personal_binary', 'idx', 'pre', 'post', 'qual', 'rot',
       'experiment_condition', 'rating', 'prompt', 'low_or_high',
       'llm_response_rot', 'prompt_condition', 'domain', 'agreement_condition',
       'pre_distance', 'post_distance', 'change_distance',
       'did_change_distance', 'did_change', 'raw_update', 'raw_ai_distance',
       'low_duration', 'usage_personal_int', 'usage_moral_int',
       'usage_convention_int', 'usage_sum', 'convince_type',
       'implied_agreement_condition'],
      dtype='object')

In [4]:
test_row = study2.iloc[3]
jas(test_row['pre'], test_row['post'], test_row['rating'])

np.float64(0.4411764705882353)

## Fitting Distributions

In [5]:
indices_more5 = [key for key, val in Counter(study2.idx).items() if val > 5]

In [6]:
trunc_study2 = study2[study2.idx.isin(indices_more5)]

In [7]:
test_a, test_b, test_loc, test_scale = beta.fit(study2[study2.idx == indices_more5[0]].pre)
test_a, test_b, test_loc, test_scale

(np.float64(1.7632791835429487),
 np.float64(0.8951377450074995),
 np.float64(-2.229494618408566),
 np.float64(101.22949461840858))

In [17]:
rng = np.random.default_rng()
test_loc + test_scale * rng.beta(test_a, test_b, size=1)

array([87.84540701])

## Let's Just Try to Build it Out

In [18]:
fraction_ai = 0.4
n = 100
T = 100

In [19]:
G = nx.erdos_renyi_graph(n=n, p=0.4)

In [20]:
node_types = ["ai" if r < fraction_ai else "human" for r in np.random.rand(n)]
type_dict = {i: t for i, t in zip(list(G.nodes), node_types)}
nx.set_node_attributes(G, type_dict, name="type")

In [21]:
test_pre = study2[study2.idx == indices_more5[0]].pre
pre_a, pre_b = fit_betabinom(n=100, mean=np.mean(test_pre), var=np.var(test_pre))
betabinom.rvs(n=100, a=pre_a, b=pre_b, size=10)

array([81, 96, 74, 37,  5, 14, 65, 97, 40, 64])

In [22]:
# randomly initialize pre
init_pre = {i: p for i, p in zip(list(G.nodes), betabinom.rvs(n=100, a=pre_a, b=pre_b, size=len(G.nodes)))}
nx.set_node_attributes(G, init_pre, name="rating")

In [23]:
for t in range(T):
    for node in G.nodes:


{0: {'type': 'human', 'rating': np.int64(57)},
 1: {'type': 'human', 'rating': np.int64(72)},
 2: {'type': 'human', 'rating': np.int64(99)},
 3: {'type': 'human', 'rating': np.int64(19)},
 4: {'type': 'human', 'rating': np.int64(46)},
 5: {'type': 'human', 'rating': np.int64(94)},
 6: {'type': 'ai', 'rating': np.int64(78)},
 7: {'type': 'human', 'rating': np.int64(92)},
 8: {'type': 'human', 'rating': np.int64(77)},
 9: {'type': 'ai', 'rating': np.int64(95)},
 10: {'type': 'human', 'rating': np.int64(39)},
 11: {'type': 'human', 'rating': np.int64(41)},
 12: {'type': 'ai', 'rating': np.int64(83)},
 13: {'type': 'ai', 'rating': np.int64(71)},
 14: {'type': 'human', 'rating': np.int64(89)},
 15: {'type': 'human', 'rating': np.int64(98)},
 16: {'type': 'human', 'rating': np.int64(8)},
 17: {'type': 'human', 'rating': np.int64(49)},
 18: {'type': 'ai', 'rating': np.int64(11)},
 19: {'type': 'human', 'rating': np.int64(62)},
 20: {'type': 'ai', 'rating': np.int64(42)},
 21: {'type': 'ai', '

In [38]:
G.nodes[0]

{'type': 'human', 'rating': np.int64(57)}

In [ ]:
event_times = []
hh_prop = 0.7
ha_prop = 0.5

hh_intervals = np.poisson(lam=1/hh_prop, size=10000)
ha_intervals = np.poisson(lam=1/ha_prop, size=10000)
np.random

while t < T:
